# 02 — ColBERT Index + Search (50-doc smoke test)

Verify RAGatouille can build a ColBERT index and run late-interaction search.
We slice 50 documents so the index builds in a few minutes even on CPU.

Full-corpus indexing and the nbits ablation are in `03_ablation_nbits.ipynb`.

In [1]:
import os, sys

# ── Auto-detect environment ───────────────────────────────────────────────────
try:
    import google.colab; _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

_IN_KAGGLE = os.path.exists("/kaggle/working")

if _IN_COLAB or _IN_KAGGLE:
    # Remote: repo will be cloned in the next cell — update REPO_URL first.
    REPO_URL  = "https://github.com/YOUR_ORG/cs410-tech-review.git"  # ← REPLACE
    REPO_ROOT = "/content/cs410-tech-review" if _IN_COLAB else "/kaggle/working/cs410-tech-review"
else:
    # Local: locate the repo root relative to this notebook.
    _here = os.path.abspath(".")
    if os.path.basename(_here) == "notebooks" and os.path.isdir(os.path.join(_here, "..", "src")):
        REPO_ROOT = os.path.abspath(os.path.join(_here, ".."))
    elif os.path.isdir(os.path.join(_here, "src")):
        REPO_ROOT = _here
    else:
        REPO_ROOT = _here  # fallback: set manually if this prints the wrong path
    REPO_URL = None
    print(f"Local mode — REPO_ROOT: {REPO_ROOT}")

Local mode — REPO_ROOT: /home/kaiyul3/cs410-tech-review


In [2]:
# Clone repo (Colab / Kaggle only — skipped automatically in local mode).
if REPO_URL and not os.path.isdir(REPO_ROOT):
    !git clone {REPO_URL} {REPO_ROOT}
elif REPO_URL:
    print(f"Repo already present at {REPO_ROOT}")
else:
    print("Local mode — skipping clone.")

Local mode — skipping clone.


In [3]:
if _IN_COLAB or _IN_KAGGLE:
    %pip install -q \
        "torch>=2.1.0,<2.4.0" \
        "transformers==4.44.2" \
        "tokenizers<0.20" \
        "faiss-cpu>=1.7.4" \
        "ragatouille==0.0.9.post2" \
        "colbert-ai>=0.2.19" \
        "langchain==0.1.20" \
        "langchain-core==0.1.53" \
        "rank_bm25>=0.2.2" \
        "beir>=2.0.0" \
        "ranx>=0.3.16" \
        "scipy>=1.11.0" \
        "numpy>=1.24.0,<2.0.0" \
        "pandas>=2.0.0" \
        "pydantic>=2.0.0" \
        "matplotlib>=3.7.0" \
        "seaborn>=0.12.0" \
        "tqdm>=4.65.0" \
        "pyyaml>=6.0" \
        "ninja"
    print("Dependencies installed.")
else:
    # Local: colbert-review conda env already provides everything.
    # Inject the env's bin dir at the head of PATH so subprocesses the kernel
    # spawns (notably ColBERT's JIT C++ extension build via ninja) resolve the
    # right ninja/g++/nvcc, regardless of how the kernel was launched.
    import sys
    _env_bin = os.path.dirname(sys.executable)
    _path = os.environ.get("PATH", "")
    if _env_bin not in _path.split(os.pathsep):
        os.environ["PATH"] = _env_bin + os.pathsep + _path
    print(f"Local mode — using existing env at {_env_bin}")

Local mode — using existing env at /home/kaiyul3/.conda/envs/colbert-review/bin


In [4]:
import sys, os
from pathlib import Path

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

RUNS_DIR    = Path(REPO_ROOT) / "results" / "runs"
METRICS_DIR = Path(REPO_ROOT) / "results" / "metrics"
FIGURES_DIR = Path(REPO_ROOT) / "results" / "figures"
DATA_DIR    = Path(REPO_ROOT) / "data" / "raw"
INDEX_ROOT  = Path(REPO_ROOT) / ".ragatouille"

for d in [RUNS_DIR, METRICS_DIR, FIGURES_DIR, DATA_DIR, INDEX_ROOT]:
    os.makedirs(d, exist_ok=True)

print("REPO_ROOT  :", REPO_ROOT)
print("RUNS_DIR   :", RUNS_DIR)
print("METRICS_DIR:", METRICS_DIR)

REPO_ROOT  : /home/kaiyul3/cs410-tech-review
RUNS_DIR   : /home/kaiyul3/cs410-tech-review/results/runs
METRICS_DIR: /home/kaiyul3/cs410-tech-review/results/metrics


In [5]:
import torch
if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"
    print("WARNING: No GPU. ColBERT indexing will be very slow.")

GPU: NVIDIA A100-SXM4-80GB


## 1. Load SciFact and slice 50 docs

In [6]:
from src.data.load_scifact import load_scifact
from src.data.schema import Corpus

full_corpus = load_scifact(split="test", raw_dir=str(DATA_DIR))

# Prioritise docs that have at least one qrel for meaningful recall numbers.
qrel_doc_ids  = {qr.doc_id for qr in full_corpus.qrels}
relevant_docs = [d for d in full_corpus.docs if d.doc_id in qrel_doc_ids]
filler_docs   = [d for d in full_corpus.docs if d.doc_id not in qrel_doc_ids]
slice_docs    = (relevant_docs + filler_docs)[:50]
slice_doc_ids = {d.doc_id for d in slice_docs}

slice_qrels   = [qr for qr in full_corpus.qrels  if qr.doc_id in slice_doc_ids]
slice_qids    = {qr.qid for qr in slice_qrels}
slice_queries = [q  for q  in full_corpus.queries if q.qid   in slice_qids]

mini_corpus = Corpus(docs=slice_docs, queries=slice_queries, qrels=slice_qrels)
mini_corpus.validate()

print(f"Mini corpus: {len(mini_corpus.docs)} docs, "
      f"{len(mini_corpus.queries)} queries, "
      f"{len(mini_corpus.qrels)} qrels")

/home/kaiyul3/.conda/envs/colbert-review/lib/python3.10/site-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm
03:52:50 INFO  src.data.load_scifact — Loading scifact split=test from /home/kaiyul3/cs410-tech-review/data/raw/scifact
03:52:50 INFO  beir.datasets.data_loader — Loading Corpus...


  0%|          | 0/5183 [00:00<?, ?it/s]

03:52:50 INFO  beir.datasets.data_loader — Loaded 5183 TEST Documents.
03:52:50 INFO  beir.datasets.data_loader — Doc Example: {'text': 'Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of the internal capsule, the mean apparent diffusion coeffic

Mini corpus: 50 docs, 62 queries, 63 qrels


## 2. Build ColBERT index (nbits=2)

In [7]:
import time
from src.colbert.index import ColBERTIndexConfig, ColBERTIndexer

index_cfg = ColBERTIndexConfig(
    checkpoint="colbert-ir/colbertv2.0",
    index_name="scifact_smoke50",
    nbits=2,
    max_document_length=256,
    index_root=str(INDEX_ROOT),
)
indexer = ColBERTIndexer(config=index_cfg)

print("Building index (downloads checkpoint on first run)...")
t0 = time.perf_counter()
index_stats = indexer.build_index(mini_corpus.docs, overwrite=True)
elapsed = time.perf_counter() - t0

print(f"\nDone in {elapsed:.1f}s")
print(f"  n_docs     : {index_stats.n_docs}")
print(f"  nbits      : {index_stats.nbits}")
print(f"  index size : {index_stats.index_bytes / 1024:.1f} KiB")
print(f"  index path : {index_stats.index_path}")

/home/kaiyul3/cs410-tech-review/src/colbert/index.py:58: UserWarning: 
********************************************************************************
RAGatouille WARNING: Future Release Notice
--------------------------------------------
RAGatouille version 0.0.10 will be migrating to a PyLate backend 
instead of the current Stanford ColBERT backend.
PyLate is a fully mature, feature-equivalent backend, that greatly facilitates compatibility.
However, please pin version <0.0.10 if you require the Stanford ColBERT backend.
********************************************************************************
  from ragatouille import RAGPretrainedModel  # lazy import


Building index (downloads checkpoint on first run)...


03:53:03 INFO  faiss.loader — Loading faiss with AVX2 support.
03:53:03 INFO  faiss.loader — Successfully loaded faiss with AVX2 support.
03:53:07 INFO  src.colbert.index — Loading ColBERT checkpoint: colbert-ir/colbertv2.0
/home/kaiyul3/.conda/envs/colbert-review/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
03:53:10 INFO  src.colbert.index — Building ColBERT index 'scifact_smoke50' (nbits=2, n_docs=50, max_doc_len=256)


---- WARNING! You are using PLAID with an experimental replacement for FAISS for greater compatibility ----
This is a behaviour change from RAGatouille 0.8.0 onwards.
This works fine for most users and smallish datasets, but can be considerably slower than FAISS and could cause worse results in some situations.
If you're confident with FAISS working on your machine, pass use_faiss=True to revert to the FAISS-using behaviour.
--------------------


[May 01, 03:53:10] #> Note: Output directory /home/kaiyul3/cs410-tech-review/.ragatouille/colbert/indexes/scifact_smoke50 already exists


[May 01, 03:53:10] #> Will delete 10 files already at /home/kaiyul3/cs410-tech-review/.ragatouille/colbert/indexes/scifact_smoke50 in 20 seconds...
[May 01, 03:53:31] [0] 		 #> Encoding 50 passages..
[May 01, 03:53:36] [0] 		 avg_doclen_est = 218.74000549316406 	 len(local_sample) = 50
[May 01, 03:53:36] [0] 		 Creating 1,024 partitions.
[May 01, 03:53:36] [0] 		 *Estimated* 10,937 embeddings.
[May 01, 03:

/home/kaiyul3/.conda/envs/colbert-review/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


[May 01, 03:54:03] [0] 		 #> Encoding 50 passages..
[May 01, 03:54:03] [0] 		 avg_doclen_est = 218.74000549316406 	 len(local_sample) = 50
[May 01, 03:54:03] [0] 		 Creating 1,024 partitions.
[May 01, 03:54:03] [0] 		 *Estimated* 10,937 embeddings.
[May 01, 03:54:03] [0] 		 #> Saving the indexing plan to /home/kaiyul3/cs410-tech-review/.ragatouille/colbert/indexes/scifact_smoke50/plan.json ..


WARNING clustering 10391 points to 1024 centroids: please provide at least 39936 training points


Clustering 10391 points in 128D to 1024 clusters, redo 1 times, 20 iterations
  Preprocessing in 0.09 s
  Iteration 19 (6.90 s, search 4.63 s): objective=2607.99 imbalance=1.486 nsplit=0       
[May 01, 03:54:11] Loading decompress_residuals_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
[May 01, 03:54:11] Loading packbits_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


OSError: CUDA_HOME environment variable is not set. Please set it to your CUDA install root.

## 3. Late-interaction search

In [ ]:
from src.colbert.search import ColBERTSearcher

searcher = ColBERTSearcher(model=indexer._model)

print(f"Searching {len(mini_corpus.queries)} queries (topk=10)...")
rankings, latency = searcher.search(mini_corpus.queries, topk=10)

print(f"\nLatency stats:")
print(f"  n_queries : {latency.n_queries}")
print(f"  total_s   : {latency.total_time_s:.2f}s")
print(f"  mean_ms   : {latency.mean_ms:.1f}ms")
print(f"  p50_ms    : {latency.p50_ms:.1f}ms")
print(f"  p95_ms    : {latency.p95_ms:.1f}ms")

Searching 62 queries (topk=10)...
Loading searcher for index scifact_smoke50 for the first time... This may take a few seconds


/home/kaiyul3/.conda/envs/colbert-review/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


[Apr 25, 03:19:07] #> Loading codec...
[Apr 25, 03:19:07] #> Loading IVF...
[Apr 25, 03:19:07] #> Loading doclens...


100%|██████████| 1/1 [00:00<00:00, 1394.38it/s]

[Apr 25, 03:19:07] #> Loading codes and residuals...



100%|██████████| 1/1 [00:00<00:00, 161.12it/s]

Searcher loaded!

#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: 5% of perinatal mortality is due to low birth weight., 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1,  1019,  1003,  1997,  2566,  3981,  9080, 13356,  2003,
         2349,  2000,  2659,  4182,  3635,  1012,   102,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')




03:19:16 INFO  src.colbert.search — ColBERT search: 62 queries, total=10.2s, p50=111.0ms, p95=198.9ms



Latency stats:
  n_queries : 62
  total_s   : 10.22s
  mean_ms   : 164.8ms
  p50_ms    : 111.0ms
  p95_ms    : 198.9ms


## 4. Qualitative inspection

In [ ]:
import pandas as pd

doc_map        = mini_corpus.doc_map()
q_map          = mini_corpus.query_map()
qrels_map_mini = mini_corpus.qrels_map()

sample_qid = mini_corpus.queries[0].qid
print(f"Query [{sample_qid}]: {q_map[sample_qid].text}\n")
print(f"Relevant docs: {list(qrels_map_mini.get(sample_qid, {}).keys())}\n")

rows = []
for rank, (doc_id, score) in enumerate(rankings.get(sample_qid, []), 1):
    doc = doc_map.get(doc_id)
    rows.append({
        "rank": rank, "doc_id": doc_id, "score": round(score, 4),
        "relevant": "YES" if doc_id in qrels_map_mini.get(sample_qid, {}) else "-",
        "title": (doc.title[:70] if doc else "N/A"),
    })
display(pd.DataFrame(rows))

Query [13]: 5% of perinatal mortality is due to low birth weight.

Relevant docs: ['1606628']



,rank,doc_id,score,relevant,title
0,1,1606628,14.7891,YES,Estimates of global prevalence of childhood un...
1,2,3898784,12.9062,-,Association of Intracerebral Hemorrhage Among ...
2,3,1180972,11.3984,-,Genetics of obesity in adult adoptees and thei...
3,4,2425364,11.0312,-,Association between maternal serum 25-hydroxyv...
4,5,1215116,9.6016,-,“Rapid-Impact Interventions”: How a Policy of ...
5,6,4687948,9.5703,-,HMG-CoA reductase inhibitors and the risk of h...
6,7,4423559,9.3359,-,Planar cell polarity signalling couples cell d...
7,8,791050,9.1875,-,The relation between past exposure to fine par...
8,9,1642727,8.9375,-,Effect of physical activity on cognitive funct...
9,10,1805641,8.8828,-,Modelling the Impact of Artemisinin Combinatio...


## 5. Write TREC run and evaluate

In [ ]:
from src.eval.metrics import write_run, evaluate_run, save_eval, DEFAULT_METRICS

run_path = RUNS_DIR / "colbert_smoke50.trec"
write_run(run_path, rankings, tag="colbert_smoke50")

qrels_map_mini = mini_corpus.qrels_map()
eval_result = evaluate_run(
    qrels_map=qrels_map_mini,
    run=rankings,
    metrics=DEFAULT_METRICS,
    run_name="colbert_smoke50",
)

print("=== Smoke-test metrics (50 docs, topk=10) ===")
for metric, value in sorted(eval_result.metrics.items()):
    print(f"  {metric:<15}: {value:.4f}")

save_eval(eval_result, METRICS_DIR / "colbert_smoke50.json")

03:19:17 INFO  src.eval.metrics — Wrote 620 lines to /home/kaiyul3/cs410-tech-review/results/runs/colbert_smoke50.trec
03:19:22 INFO  src.eval.metrics — Saved metrics to /home/kaiyul3/cs410-tech-review/results/metrics/colbert_smoke50.json


=== Smoke-test metrics (50 docs, topk=10) ===
  map            : 0.8995
  mrr@10         : 0.8995
  ndcg@10        : 0.9163
  recall@100     : 0.9677


## 6. Index-size file breakdown

In [ ]:
from pathlib import Path

index_path = Path(index_stats.index_path)
if index_path.exists():
    files     = sorted(index_path.rglob("*"))
    rows      = [{"file": str(f.relative_to(index_path)), "size_kb": f.stat().st_size / 1024}
                 for f in files if f.is_file()]
    total_kib = sum(r["size_kb"] for r in rows)
    display(pd.DataFrame(rows).sort_values("size_kb", ascending=False).head(15))
    print(f"\nTotal: {total_kib:.1f} KiB across {len(rows)} files")
else:
    print(f"Index path not found: {index_path}")

,file,size_kb
2,0.residuals.pt,684.734375
5,centroids.pt,257.162109
11,plan.json,79.002930
9,metadata.json,78.461914
6,collection.json,76.604492
0,0.codes.pt,43.839844
8,ivf.pid.pt,23.898438
4,buckets.pt,1.398438
3,avg_residual.pt,1.176758
10,pid_docid_map.json,0.864258



Total: 1247.4 KiB across 12 files


## Summary

- ColBERT checkpoint `colbert-ir/colbertv2.0` indexed 50 SciFact docs.
- Late-interaction search confirmed working; latency numbers above.
- Full-corpus indexing and the nbits ablation are in notebook 03.